# OpenSearch Anomaly Detection Tutorial

## Setup and Installation

In [ ]:
!pip3 install opensearch-py

In [ ]:
from opensearchpy import OpenSearch
from opensearchpy import OpenSearch, helpers
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Run OpenSearch Container

```bash

docker run -d -p 9200:9200 -p 9600:9600 -e "discovery.type=single-node" -e "OPENSEARCH_INITIAL_ADMIN_PASSWORD=YourStrongPassword123" opensearchproject/opensearch:latest

```

## Connect to OpenSearch

In [ ]:
client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "YourStrongPassword123"),
    use_ssl=True,
    verify_certs=False
)

## Create Index

In [ ]:
client.indices.delete(index="transactions", ignore=[400, 404])

In [ ]:
client.indices.create(
    index='transactions',
    body={
        'mappings': {
            'properties': {
                'timestamp': {'type': 'date'},
                'amount': {'type': 'float'}
            }
        }
    }
)

## Generate and Ingest Sample Data

In [ ]:
from datetime import datetime, timedelta
import random

base_time = datetime.utcnow() - timedelta(minutes=500)
documents = []

for i in range(500):
    amount = random.randint(95, 110)
    if i in [420, 455, 490]:
        amount = 5000
    documents.append({
        "timestamp": (base_time + timedelta(minutes=i)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "amount": amount
    })


actions = [{"_index": "transactions", "_source": doc} for doc in documents]
helpers.bulk(client, actions)
client.indices.refresh(index="transactions")


print("Inserted", len(documents), "documents")
print("Sample data ingested successfully.")

## Create Anomaly Detector

In [ ]:
detector_config = {
    "name": "transaction-detector",
    "description": "Detect anomalies in transaction amount",
    "time_field": "timestamp",
    "indices": ["transactions"],
    "filter_query": {
        "bool": {
            "filter": []
        }
    },
    "feature_attributes": [
        {
            "feature_name": "avg_amount",
            "feature_enabled": True,
            "aggregation_query": {
                "avg_amount": {
                    "avg": {
                        "field": "amount"
                    }
                }
            }
        }
    ],
    "detection_interval": {
        "period": {
            "interval": 1,
            "unit": "Minutes"
        }
    },
    "window_delay": {
        "period": {
            "interval": 1,
            "unit": "Minutes"
        }
    },
    "shingle_size": 8
}

response = client.transport.perform_request(
    method='POST',
    url='/_plugins/_anomaly_detection/detectors',
    body=detector_config
)

print(response)
detector_id = response['_id']
print(f"Detector created successfully with ID: {detector_id}")

## Start Detector

In [ ]:
response = client.transport.perform_request(
    method='POST',
    url=f'/_plugins/_anomaly_detection/detectors/{detector_id}/_start'
)
print(f"Detector started successfully: {response}")

## View Detection Results

In [ ]:
response = client.transport.perform_request(
    method='GET',
    url=f'/_plugins/_anomaly_detection/detectors/{detector_id}/results'
)

print(f"Anomaly detection results: {response}")

## Run OpenSearch Dashboard Container

```bash
podman run -d \
  --name opensearch-dashboards \
  -p 5601:5601 \
  -e 'OPENSEARCH_HOSTS=["https://localhost:9200"]' \
  docker.io/opensearchproject/opensearch-dashboards:latest
```

![alt text](detector-screenshot.png)